In [3]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv")
subscriptions = pd.read_csv("../data/raw/subscriptions.csv")
revenue = pd.read_csv("../data/raw/revenue.csv")

In [4]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['churn_date'] = pd.to_datetime(customers['churn_date'])

subscriptions['month'] = pd.to_datetime(subscriptions['month'])
revenue['month'] = pd.to_datetime(revenue['month'])


In [5]:
#First subscription month per cusomer
first_subscription =(
    subscriptions
    .groupby('customer_id')['month']
    .min()
    .reset_index(name ='first_subscription_month')
)
activation = customers.merge(
    first_subscription,
    on='customer_id',
    how='left'
)
activation['days_to_activate']=(
    activation['first_subscription_month'] - activation['signup_date']
).dt.days


In [6]:
activation['signup_month'] = activation['signup_date'].dt.to_period('M')
activation['first_subscription_month'] = activation['first_subscription_month'].dt.to_period('M')


In [7]:
activation['months_to_activate'] = (
    activation['first_subscription_month'] - activation['signup_month']
)

activation['months_to_activate'] = activation['months_to_activate'].apply(
    lambda x: x.n if pd.notnull(x) else None
)

In [8]:
activation['months_to_activate'].describe()

count    168.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: months_to_activate, dtype: float64

In [9]:
activation['months_to_activate'].value_counts().sort_index()


months_to_activate
0.0    168
Name: count, dtype: int64

In [10]:
activation_rate = activation['first_subscription_month'].notna().mean()
activation_rate


np.float64(0.168)

In [11]:
month_0_rate = (activation['months_to_activate'] == 0).mean()
month_0_rate


np.float64(0.168)

## Activation Insight
All activated customers converted in the same month as signup, indicating that activation is immediate rather than gradual. Customers who fail to activate in their signup month rarely convert later, making early onboarding and first-month experience critical for long-term retention and revenue.
